# 07 — Save model + prediction API

**Goal:** v3 becomes `models/v3.pkl` + `models/v3_metadata.json`.\
Then `src/solarops/predict.py` provides `load_model()`, `predict()`.\
Test: load from disk, predict on 5 rows, match in-memory output exactly.

## Setup — reproduce v3

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path
import json
import joblib
import pandas as pd
from lightgbm import LGBMRegressor

import solarops.pipeline
from solarops.pipeline import (
    load_raw, align_15m, split_time, impute_features, evaluate,
    add_time_features, FEATURES, TARGET_COL,
)

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR   = PROJECT_ROOT / "data/raw/pvdaq/2023-solar-data-prize/system-2107"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

raw = load_raw(DATA_DIR)
df  = align_15m(raw)
train, test = split_time(df, "2022-07-01")
X_train, X_test, y_train, y_test = impute_features(train, test)

TIME_FEATURES = ["hour", "month", "dayofyear"]
train_fe = add_time_features(train); test_fe = add_time_features(test)
X_train_v3 = pd.concat([X_train, train_fe[TIME_FEATURES]], axis=1)
X_test_v3  = pd.concat([X_test,  test_fe[TIME_FEATURES]],  axis=1)

v3 = LGBMRegressor(random_state=42, n_estimators=300, learning_rate=0.05).fit(X_train_v3, y_train)
print(evaluate(v3, X_test_v3, y_test, "v3"))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000711 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1047
[LightGBM] [Info] Number of data points in the train set: 160396, number of used features: 6
[LightGBM] [Info] Start training from score 153.684736
{'model': 'v3', 'MAE': 25.68962829665161, 'RMSE': 50.784879138540894, 'R2': 0.9518473724595655}


## Step 1 — save model + metadata

Why metadata? A `.pkl` alone doesn't say **what version**, **what features**, **what date it was trained on**.\
In 6 months you'll need this. Save it next to the model.

In [2]:
v3_metrics = evaluate(v3, X_test_v3, y_test, "v3")

joblib.dump(v3, MODELS_DIR / "v3.pkl")

metadata = {
    "version": "v3",
    "features": list(X_train_v3.columns),
    "target": TARGET_COL,
    "split_date": "2022-07-01",
    "train_range": [str(train["measured_on"].min()), str(train["measured_on"].max())],
    "test_range":  [str(test["measured_on"].min()),  str(test["measured_on"].max())],
    "metrics":     {k: round(v, 4) for k, v in v3_metrics.items() if k != "model"},
    "trained_at":  pd.Timestamp.now().isoformat(timespec="seconds"),
    "data": "PVDAQ System 2107",
}
(MODELS_DIR / "v3_metadata.json").write_text(json.dumps(metadata, indent=2))

print("saved:")
print(" ", MODELS_DIR / "v3.pkl")
print(" ", MODELS_DIR / "v3_metadata.json")
print("\nmetadata:", json.dumps(metadata, indent=2)[:400], "...")

saved:
  /Users/apple/Desktop/SolarOps/models/v3.pkl
  /Users/apple/Desktop/SolarOps/models/v3_metadata.json

metadata: {
  "version": "v3",
  "features": [
    "poa_irradiance_o_149574",
    "ambient_temperature_o_149575",
    "wind_speed_o_149576",
    "hour",
    "month",
    "dayofyear"
  ],
  "target": "meter_revenue_grade_ac_output_meter_149578",
  "split_date": "2022-07-01",
  "train_range": [
    "2017-12-01 00:00:00",
    "2022-06-30 23:45:00"
  ],
  "test_range": [
    "2022-07-01 00:00:00",
    "2023-10- ...


## Step 2 — write `src/solarops/predict.py`

Two functions:
- `load_model(path)` — read pkl + metadata, return both.
- `predict(df, model, feature_columns)` — df must already have all features; return predictions.

## Step 3 — test: load from disk, predict, compare to in-memory

In [3]:
import importlib; importlib.reload(solarops.pipeline)
import solarops.predict; importlib.reload(solarops.predict)
from solarops.predict import load_model, predict

model_loaded, meta = load_model(MODELS_DIR / "v3.pkl")
print("loaded version:", meta["version"])
print("expected MAE :", meta["metrics"]["MAE"])

pred_loaded = predict(X_test_v3.head(5), model_loaded, meta["features"])
pred_mem    = v3.predict(X_test_v3.head(5))

import numpy as np
assert np.allclose(pred_loaded.values, pred_mem), "loaded model disagrees with in-memory!"
print("\nTop-5 predictions from loaded model:")
print(pred_loaded.round(2))

loaded version: v3
expected MAE : 25.6896

Top-5 predictions from loaded model:
160396    0.42
160397    0.42
160398    0.42
160399    0.42
160400    0.42
Name: predicted, dtype: float64


## Step 4 — simulate a real prediction request

Given 3 hypothetical new readings, produce predictions + trust label.
This is what a production API call would look like.

In [8]:
new_readings = pd.DataFrame({
    "measured_on": pd.to_datetime(["2024-06-15 12:00", "2024-06-15 12:15", "2024-06-15 12:30"]),
    "poa_irradiance_o_149574":      [850.0, 870.0, None],
    "ambient_temperature_o_149575": [32.0,  33.0,  34.0],
    "wind_speed_o_149576":          [3.5,   3.2,   3.0],
})

from solarops.pipeline import IRRADIANCE_COL, FEATURES

# --- Step A: predict() on RAW input (POA NaN still present) ---
# Must add the required time features so the only missing thing is POA
new_readings["hour"]      = new_readings["measured_on"].dt.hour
new_readings["month"]     = new_readings["measured_on"].dt.month
new_readings["dayofyear"] = new_readings["measured_on"].dt.day_of_year

print("=== predict() with reject_missing_poa=True (default) ===")
try:
    predict(new_readings, model_loaded, meta["features"])
    print("no error (unexpected!)")
except ValueError as e:
    print("✅ correctly refused:", e)

=== predict() with reject_missing_poa=True (default) ===
✅ correctly refused: 1 row(s) have missing POA irradiance. Handle missing input upstream, or pass reject_missing_poa=False.


In [9]:
# --- Step B: caller imputes upstream, then asks predict to run anyway ---
new_readings["poa_was_missing"] = new_readings[IRRADIANCE_COL].isna()

for feat in FEATURES:
    new_readings[feat] = new_readings[feat].fillna(X_train[feat].median())

new_readings["trust"] = [
    "unreliable_input" if (was_missing and 6 <= h <= 20) else "ok"
    for was_missing, h in zip(new_readings["poa_was_missing"], new_readings["hour"])
]

new_readings["predicted"] = predict(
    new_readings, model_loaded, meta["features"], reject_missing_poa=False
).values

print(new_readings[["measured_on", IRRADIANCE_COL, "predicted", "trust"]])

          measured_on  poa_irradiance_o_149574   predicted             trust
0 2024-06-15 12:00:00               850.000000  510.562219                ok
1 2024-06-15 12:15:00               870.000000  505.441871                ok
2 2024-06-15 12:30:00                 6.566667  269.239311  unreliable_input


## What 07 gives us

- `models/v3.pkl` + `v3_metadata.json` — model is portable, versioned, has metrics + features.
- `src/solarops/predict.py` — `load_model()` and `predict()` usable from any notebook, script, or future API.
- Tested: fresh load reproduces in-memory predictions exactly.

**Stage 3 (prediction workflow) is now unblocked.** Any future work calls `load_model("models/v3.pkl")`, never retrains.

## What's NOT here yet
- Automated retraining
- Input validation (POA > 2000, negative temp)
- API endpoint (FastAPI/Flask) — Stage 4
- Model versioning beyond filename (no MLflow yet)